# Data Cleaning & Quality Assessment Project

**Problem Statement:** Real-world datasets often contain missing, inconsistent, and noisy data. This notebook focuses on cleaning and preparing a raw insurance/healthcare dataset for downstream analysis and modeling.

### Objective:
- Identify missing, duplicate, and inconsistent values.
- Apply appropriate cleaning techniques (imputation, encoding alignment).
- Handle outliers systematically.
- Document data quality issues and resolutions.

In [10]:
import pandas as pd
import numpy as np

# Configure display options for better DataFrame printing
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## 1. Load the Dataset
We will read the dataset from the provided path `/content/skillairo.csv` and inspect its shape and schema.

In [2]:
# Load dataset
try:
    df = pd.read_csv('/content/skillairo.csv')
    print("Dataset loaded successfully!")
except FileNotFoundError:
    # Fallback placeholder if file is not immediately placed, creating a sample simulation structured like Kaggle Insurance
    print("File not found. Creating a simulation of the Insurance dataset for demonstration.")
    np.random.seed(42)
    n_samples = 1000
    df = pd.DataFrame({
        'age': np.random.choice([19, 28, 33, 45, 50, 64, None], size=n_samples, p=[0.15, 0.15, 0.20, 0.20, 0.15, 0.10, 0.05]),
        'sex': np.random.choice(['female', 'male', 'FEMALE', 'MALE', None], size=n_samples, p=[0.45, 0.45, 0.04, 0.04, 0.02]),
        'bmi': np.random.normal(loc=30, scale=6, size=n_samples),
        'children': np.random.choice([0, 1, 2, 3, 4, 5], size=n_samples),
        'smoker': np.random.choice(['yes', 'no', 'YES', 'NO'], size=n_samples, p=[0.18, 0.78, 0.02, 0.02]),
        'region': np.random.choice(['southwest', 'southeast', 'northwest', 'northeast'], size=n_samples),
        'charges': np.random.exponential(scale=13000, size=n_samples) + 1000
    })
    # Introduce some outliers and missingness intentionally
    df.loc[df.sample(frac=0.03).index, 'bmi'] = np.nan
    df.loc[df.sample(frac=0.01).index, 'bmi'] = 99.9  # extreme value outlier

display(df.head())
print(f"Dataset dimensions: {df.shape[0]} rows, {df.shape[1]} columns")

Dataset loaded successfully!


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


Dataset dimensions: 1338 rows, 7 columns


## 2. Quality Assessment
Let us check for overall information, missing values, duplicates, and unique categories to find inconsistencies.

In [3]:
# Schema information
print("=== Dataset Info ===")
df.info()

# Missing values summary
print("\n=== Missing Values Per Column ===")
missing_info = df.isnull().sum()
display(missing_info[missing_info > 0])

# Duplicate rows
duplicates = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {duplicates}")

=== Dataset Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB

=== Missing Values Per Column ===


,0



Number of duplicate rows: 1


### Inspect Categorical Inconsistencies
Real-world datasets often have capitalization inconsistencies (e.g., 'female' vs 'FEMALE', 'yes' vs 'YES'). Let's investigate the unique categories.

In [4]:
for col in df.select_dtypes(include=['object']).columns:
    print(f"Unique values in '{col}': {df[col].unique()}")

Unique values in 'sex': ['female' 'male']
Unique values in 'smoker': ['yes' 'no']
Unique values in 'region': ['southwest' 'southeast' 'northwest' 'northeast']


## 3. Data Cleaning
We will standardize spelling/casing, handle missing records, and drop duplicates.

In [5]:
# 1. Harmonize casing for categorical variables
if 'sex' in df.columns:
    df['sex'] = df['sex'].astype(str).str.lower().replace('nan', np.nan)
if 'smoker' in df.columns:
    df['smoker'] = df['smoker'].astype(str).str.lower().replace('nan', np.nan)

# 2. Drop duplicates if any
if duplicates > 0:
    df = df.drop_duplicates()
    print(f"Dropped {duplicates} duplicate entries.")

# 3. Handle missing values
# Impute categorical nulls with mode and numerical nulls with median
if df['age'].isnull().any():
    df['age'] = df['age'].fillna(df['age'].median())
if df['bmi'].isnull().any():
    df['bmi'] = df['bmi'].fillna(df['bmi'].median())
if 'sex' in df.columns and df['sex'].isnull().any():
    df['sex'] = df['sex'].fillna(df['sex'].mode()[0])

print("Post-cleaning categories verification:")
for col in df.select_dtypes(include=['object']).columns:
    print(f"Unique values in '{col}': {df[col].unique()}")

Dropped 1 duplicate entries.
Post-cleaning categories verification:
Unique values in 'sex': ['female' 'male']
Unique values in 'smoker': ['yes' 'no']
Unique values in 'region': ['southwest' 'southeast' 'northwest' 'northeast']


## 4. Outlier Assessment & Handling
Let's analyze numerical distributions (especially `bmi` and `charges`) using boxplots to visualize outliers.

In [11]:
# Identify outliers using pandas descriptive statistics (Quantiles)
summary_stats = df[['bmi', 'charges']].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
display(summary_stats)

# Calculate bounds to show potential outliers in numerical columns
for col in ['bmi', 'charges']:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"Column '{col}': found {len(outliers)} outliers outside range ({lower:.2f}, {upper:.2f})")

,bmi,charges
count,1337.000000,1337.000000
mean,30.663452,13279.121487
std,6.100468,12110.359656
min,15.960000,1121.873900
25%,26.290000,4746.344000
50%,30.400000,9386.161300
75%,34.700000,16657.717450
90%,38.626000,34832.741600
95%,41.108000,41210.049800
99%,46.411200,48537.796878


Column 'bmi': found 9 outliers outside range (13.67, 47.32)
Column 'charges': found 139 outliers outside range (-13120.72, 34524.78)


### Capping Outliers using the IQR (Interquartile Range) Method

In [12]:
# Helper function to cap outliers using IQR (Pandas/NumPy)
def cap_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    return np.clip(series, lower_bound, upper_bound)

# Capping extreme bmi values
df['bmi_cleaned'] = cap_outliers(df['bmi'])

# Verify the metrics before and after capping
comparison = pd.DataFrame({
    'Before Capping': df['bmi'].describe(),
    'After Capping': df['bmi_cleaned'].describe()
})
display(comparison)

,Before Capping,After Capping
count,1337.000000,1337.000000
mean,30.663452,30.650247
std,6.100468,6.059655
min,15.960000,15.960000
25%,26.290000,26.290000
50%,30.400000,30.400000
75%,34.700000,34.700000
max,53.130000,47.315000


## 5. Final Dataset Quality Overview

In [8]:
print("Missing values left:")
print(df.isnull().sum())
print("\nFirst 5 rows of final cleaned dataset:")
display(df.head())

Missing values left:
age            0
sex            0
bmi            0
children       0
smoker         0
region         0
charges        0
bmi_cleaned    0
dtype: int64

First 5 rows of final cleaned dataset:


,age,sex,bmi,children,smoker,region,charges,bmi_cleaned
0,19,female,27.900,0,yes,southwest,16884.92400,27.900
1,18,male,33.770,1,no,southeast,1725.55230,33.770
2,28,male,33.000,3,no,southeast,4449.46200,33.000
3,33,male,22.705,0,no,northwest,21984.47061,22.705
4,32,male,28.880,0,no,northwest,3866.85520,28.880


## 6. Export Cleaned Dataset
Save the cleaned, structured, and outlier-capped dataset to a new CSV file for analysis and modeling.

In [9]:
# Export the cleaned dataframe to a new CSV file
output_path = '/content/skillairo_cleaned.csv'
df.to_csv(output_path, index=False)

import os
if os.path.exists(output_path):
    print(f"Success! Cleaned dataset exported to: {output_path}")
    print(f"File size: {os.path.getsize(output_path) / 1024:.2f} KB")
else:
    print("Error: Failed to export the dataset.")

Success! Cleaned dataset exported to: /content/skillairo_cleaned.csv
File size: 60.92 KB
